In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine
from src.curves.projection_curve import ProjectionCurve
from src.curves.zero_curve import ZeroCurve

from src.instruments.instrument_builder import InstrumentBuilder

from src.trades.interest_rate_swap import InterestRateSwap

from src.pricing.swap_pricer import SwapPricer

from src.risk.ir_risk_engine import IRRiskEngine
from src.risk.portfolio_risk_engine import PortfolioRiskEngine

from src.portfolio.portfolio import Portfolio

In [2]:
# downloading market curves
market_loader = MarketLoader()
market_curves = market_loader.market_loader_pipeline()

# downloading swap curves
swap_loader = MarketLoader()
swap_curves = swap_loader.swap_loader_pipeline()

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..
usd_ois curve dataset already downloaded..
eur_ois curve dataset already downloaded..


In [3]:
### create curve snapshots
# SOFR snapshot
sofr_df = market_curves['sofr']

latest_date = sofr_df.index[-1]
latest_sofr_curve = sofr_df.iloc[-1]

sofr_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'sofr',
    as_of_date = latest_date,
    curve_row = latest_sofr_curve
)

# Futures snapshot
future_df = market_curves['futures']
latest_futures_curve = future_df.iloc[-1]

futures_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'futures',
    as_of_date = latest_date,
    curve_row = latest_futures_curve
)

# OIS snapshot
ois_df = swap_curves['usd_ois']

latest_swap_date = ois_df.index[-1]
latest_swap_curve = ois_df.iloc[-1]

ois_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'usd_ois',
    as_of_date = latest_swap_date,
    curve_row = latest_swap_curve
)

In [4]:
### create instruments from curve snapshot
# deposits
deposit_instruments = InstrumentBuilder.build_deposit_instruments(snapshot = sofr_snapshot)

# futures
future_instruments = InstrumentBuilder.build_future_instruments(snapshot = futures_snapshot)

# ois
ois_instruments = InstrumentBuilder.build_ois_instruments(snapshot = ois_snapshot)

### discount, projection and zero curve builder
# bootstrapping engine for generating the discount curve
all_instruments = deposit_instruments + future_instruments + ois_instruments

engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = all_instruments
)

# projection curve
projection_curve = ProjectionCurve(discount_curve = discount_curve)

# zero curve
zero_curve = ZeroCurve(discount_curve = discount_curve)

In [5]:
### construct swap portfolio
# sample IRS trade objects
IR_swap_1 = InterestRateSwap(
    notional = 1_000_000,
    maturity = 3.0,
    fixed_rate = 3.70,
    pay_fixed = True
)

IR_swap_2 = InterestRateSwap(
    notional = 3_750_000,
    maturity = 10.0,
    fixed_rate = 3.15,
    pay_fixed = False
)

IR_swap_3 = InterestRateSwap(
    notional = 5_300_000,
    maturity = 5.0,
    fixed_rate = 3.65,
    pay_fixed = True
)

swap_list = [IR_swap_1, IR_swap_2, IR_swap_3]

# swap portfolio
IRS_portfolio = Portfolio()

for swap in swap_list:
    IRS_portfolio.add_trade(trade = swap)

IRS_portfolio.summary()

[InterestRateSwap(Notional=1000000,  Maturity=3.0,  FixedRate=3.7,  Direction=PAY_FIXED),
 InterestRateSwap(Notional=3750000,  Maturity=10.0,  FixedRate=3.15,  Direction=RECEIVE_FIXED),
 InterestRateSwap(Notional=5300000,  Maturity=5.0,  FixedRate=3.65,  Direction=PAY_FIXED)]

In [6]:
### portfolio-level risk engine
# pricer
pricer = SwapPricer(
    discount_curve = discount_curve,
    projection_curve = projection_curve
)

# ir risk engine
risk_engine = IRRiskEngine(
    pricer = pricer,
    zero_curve = zero_curve
)

# portfolio risk engine
portfolio_risk = PortfolioRiskEngine(ir_risk_engine = risk_engine)

portfolio_risk.portfolio_risk_report(
    portfolio = IRS_portfolio
)

,Metric,Value
0,Portfolio PV,-1.067350e+05
1,Portfolio PV01,-4.200495e+02
2,Portfolio DV01,4.200495e+02
3,Portfolio Convexity,1.725637e+08
